<a href="https://colab.research.google.com/github/Jashwanth248/Portfolio/blob/main/Copy_of_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from datasets import load_dataset
import numpy as np
import torchvision.transforms as transforms

Load MNIST

In [ ]:
dataset = load_dataset("ylecun/mnist")

train_ds = dataset["train"]
test_ds = dataset["test"]

Convert to Tensor

In [ ]:
def transform(example):
    image = np.array(example["image"]) / 255.0
    image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
    label = torch.tensor(example["label"], dtype=torch.long)
    return {"image": image, "label": label}

train_ds = train_ds.map(transform)
test_ds = test_ds.map(transform)

train_ds.set_format(type="torch", columns=["image", "label"])
test_ds.set_format(type="torch", columns=["image", "label"])

DataLoad

In [ ]:
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=1000, shuffle=False)

Data Augmentation

In [ ]:
augment = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(0, translate=(0.1, 0.1))
])

MLP Model

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.net(x)

CNN Model

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        return self.fc(x)

Transformer Model

In [ ]:
class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed_dim = 64

        self.patch_embed = nn.Conv2d(1, self.embed_dim, kernel_size=7, stride=7)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.embed_dim,
            nhead=4,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(self.embed_dim, 10)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x.flatten(2).permute(0, 2, 1)

        x = self.transformer(x)
        x = x.mean(dim=1)

        return self.fc(x)

Train Function (WITH AUGMENTATION)

In [ ]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for batch in loader:
        data = batch["image"]
        target = batch["label"]


        data = torch.stack([augment(img) for img in data])

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

Test Function

In [ ]:
def test(model, loader):
    model.eval()
    correct = 0

    with torch.no_grad():
        for batch in loader:
            data = batch["image"]
            target = batch["label"]

            output = model(data)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()

    return 100 * correct / len(loader.dataset)

Train CNN

In [ ]:
print("\nTraining CNN Model...")
model = CNN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

cnn_acc = 0

for epoch in range(5):
    loss = train(model, train_loader, optimizer, criterion)
    cnn_acc = test(model, test_loader)
    print(f"Epoch {epoch+1}: Loss={loss:.4f}, Acc={cnn_acc:.2f}%")

Train CNN

In [ ]:
print("\nTraining MLP Model...")
model = MLP()
optimizer = optim.Adam(model.parameters(), lr=0.001)

mlp_acc = 0

for epoch in range(5):
    loss = train(model, train_loader, optimizer, criterion)
    mlp_acc = test(model, test_loader)
    print(f"Epoch {epoch+1}: Loss={loss:.4f}, Acc={mlp_acc:.2f}%")

Train Transformer

In [ ]:
print("\nTraining Transformer Model...")
model = TransformerModel()
optimizer = optim.Adam(model.parameters(), lr=0.001)

trans_acc = 0

for epoch in range(5):
    loss = train(model, train_loader, optimizer, criterion)
    trans_acc = test(model, test_loader)
    print(f"Epoch {epoch+1}: Loss={loss:.4f}, Acc={trans_acc:.2f}%")

Final Comparison

In [ ]:
print("\n=== Final Model Comparison ===")
print(f"MLP Accuracy        : {mlp_acc:.2f}%")
print(f"CNN Accuracy        : {cnn_acc:.2f}%")
print(f"Transformer Accuracy: {trans_acc:.2f}%")